# ERC with per-index implied volatility and portfolio volatility targeting

This is a detailed, executable walkthrough of
[erc_vol_targetting.md](../../docs/research/erc_vol_targetting.md). The guide's
filename uses `targetting`; this notebook uses the requested `targeting` spelling.

The sequence is **historical returns → shrunk correlation → IV-based forward
covariance → fully invested ERC weights → total exposure and cash**.

For a portfolio of global equity indices, ERC allocates risk *between* indices.
Volatility targeting adjusts how much equity risk the portfolio takes overall.
Neither a larger ticker count nor equal risk shares removes common equity-market
risk. This matters in your example: normal-year realized volatility around 8%,
and crisis-year volatility around 14–18%, even with inverse-volatility sizing.

Run all cells with the repository's `venv` Python kernel. Everything runs on
explicitly **synthetic teaching fixtures**, with no API keys, market downloads,
calibration fit, or trading. The 8% target and no-leverage policy are illustrations
from the guide, not recommended defaults. The final data cell lets you provide
your own prices, forward forecasts, and policy choices.

## 1. Load the existing implementation

Import the functions from the research recipe instead of maintaining another
solver in this notebook. Repository discovery works when Jupyter starts in the
repository root or in `cmd/notebooks`. Plotly figures contain their data in the
notebook and use the notebook viewer's Plotly renderer; no chart data download
is needed.

In [1]:
import sys
import inspect
from pathlib import Path

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'docs/research/risk_parity_recipe.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import Code, display
from docs.research.risk_parity_recipe import (
    estimate_covariance, forward_covariance, equal_risk_weights, volatility_target,
)

pio.renderers.default = 'plotly_mimetype'
print('Python:', sys.executable)
print('Recipe:', ROOT / 'docs/research/risk_parity_recipe.py')

Python: /Users/constantinpavlicu/dev/apollo3/venv/bin/python
Recipe: /Users/constantinpavlicu/dev/apollo3/docs/research/risk_parity_recipe.py


## 2. Build a small price and return panel whose arithmetic is visible

There are three generic indices. In the *constructed* correlation matrix, A and
B have correlation 0.9 and C is uncorrelated with them. This is the earlier
analytical example, **not a claim about actual global equity correlations**.

Paired positive/negative Cholesky rows produce a zero-mean return panel with that
sample correlation. Scaling the rows creates differing historical volatilities.
The panel has only the observations needed for this arithmetic fixture; using
all of them is not a recommended estimation lookback. Prices start at an arbitrary
100 units, and all returns stay above −100%.

Use adjusted total-return prices in a common currency. `pct_change(fill_method=None)`
produces simple decimal returns and does not fill missing observations. A common
calendar alone does not solve asynchronous closes: actual timestamps must also
be consistent with when the information became available.

In [2]:
assets = pd.Index(['Index A', 'Index B', 'Index C'], name='index')
constructed_correlation = pd.DataFrame(
    [[1, 0.9, 0], [0.9, 1, 0], [0, 0, 1]], index=assets, columns=assets,
)
rows = np.linalg.cholesky(constructed_correlation.to_numpy()).T
rows = rows * np.array([1, 2, 3]) / 100  # Synthetic return units, not policy.
synthetic_returns = np.vstack([rows, -rows])
levels = np.vstack([
    np.ones(len(assets)), np.cumprod(1 + synthetic_returns, axis=0),
]) * 100
dates = pd.bdate_range('2026-09-01', periods=len(levels))
total_return_prices = pd.DataFrame(levels, index=dates, columns=assets)
signal_date = dates[-1]
correlation_lookback = len(synthetic_returns)  # Entire fixture, not a default.
returns = total_return_prices.pct_change(fill_method=None).iloc[1:]

assert np.allclose(returns, synthetic_returns)
assert (total_return_prices > 0).all().all()
display(total_return_prices)
display(returns.style.format('{:.3%}'))

index,Index A,Index B,Index C
2026-09-01,100.00,100.000000,100.00
2026-09-02,101.00,101.800000,100.00
2026-09-03,101.00,102.687472,100.00
2026-09-04,101.00,102.687472,103.00
2026-09-07,99.99,100.839097,103.00
2026-09-08,99.99,99.960002,103.00
2026-09-09,99.99,99.960002,99.91


index,Index A,Index B,Index C
2026-09-02 00:00:00,1.000%,1.800%,0.000%
2026-09-03 00:00:00,0.000%,0.872%,0.000%
2026-09-04 00:00:00,0.000%,0.000%,3.000%
2026-09-07 00:00:00,-1.000%,-1.800%,0.000%
2026-09-08 00:00:00,0.000%,-0.872%,0.000%
2026-09-09 00:00:00,0.000%,0.000%,-3.000%


## 3. Estimate correlation, with shrinkage chosen from the data

For centered returns $r_{i,t}-\bar r_i$, divide by each asset's population
standard deviation (`ddof=0`, matching scikit-learn). Ledoit–Wolf on these unit-
variance series shrinks their empirical correlation toward the identity:

$$\widehat C=(1-\delta)C_{\mathrm{sample}}+\delta I.$$

There is no hand-selected shrinkage coefficient. The tiny synthetic panel can
produce substantial shrinkage: do not interpret its fitted intensity as a
parameter for your real portfolio. This standardized-return variant is a modeling
choice, not an optimality guarantee for the later IV-rescaled covariance.
[Estimator documentation](https://scikit-learn.org/1.8/modules/generated/sklearn.covariance.LedoitWolf.html)

The existing estimator first returns historical covariance. Dividing by its
diagonal standard deviations recovers the correlation used by the forward model.

In [3]:
display(Code(inspect.getsource(estimate_covariance), language='python'))
historical_covariance, shrinkage = estimate_covariance(returns)
historical_vol = np.sqrt(np.diag(historical_covariance))
shrunk_correlation = pd.DataFrame(
    historical_covariance.to_numpy() / np.outer(historical_vol, historical_vol),
    index=assets, columns=assets,
)
sample_correlation = returns.corr()
assert np.allclose(sample_correlation, constructed_correlation)
assert np.allclose(np.diag(historical_covariance), returns.var(ddof=0))
assert np.allclose(
    shrunk_correlation,
    (1 - shrinkage) * sample_correlation.to_numpy() + shrinkage * np.eye(len(assets)),
)
print('Fitted shrinkage intensity:', shrinkage)
display(pd.DataFrame({
    'historical_return_vol': historical_vol,
    'historical_return_variance': np.diag(historical_covariance),
}, index=assets))

figure = make_subplots(rows=1, cols=2, subplot_titles=['Sample correlation', 'Shrunk correlation'])
for column, matrix in enumerate([sample_correlation, shrunk_correlation], start=1):
    figure.add_trace(go.Heatmap(
        z=matrix.to_numpy(), x=assets.tolist(), y=assets.tolist(),
        zmin=-1, zmax=1, colorscale='RdBu', showscale=(column == 2),
        hovertemplate='%{y} / %{x}: %{z:.3f}<extra></extra>',
    ), row=1, col=column)
figure.update_layout(title='Shrinkage changes correlation; historical variances are preserved')
figure.show()

def estimate_covariance(returns):
    """Return variance-preserving shrunk covariance and fitted intensity."""
    if not isinstance(returns, pd.DataFrame):
        raise TypeError("returns must be a pandas DataFrame")
    if returns.shape[0] < 2 or returns.shape[1] == 0:
        raise ValueError("Need at least two observations and one asset")
    if (not returns.columns.is_unique or not returns.index.is_unique
            or not returns.index.is_monotonic_increasing):
        raise ValueError("Asset labels must be unique; dates unique and sorted")
    values = returns.to_numpy(dtype=float)
    if not np.isfinite(values).all():
        raise ValueError("Use a complete, finite, aligned return panel")
    centered = values - values.mean(axis=0)
    volatility = centered.std(axis=0, ddof=0)
    if not np.isfinite(volatility).all() or (volatility <= 0).any():
        raise ValueError("Every asset must have positive finite volatility")
    fitted = LedoitWolf(assume_centered=True, store_precision=False).fit(
        centered / volatility
    )
    covariance = fitted.covariance_ * np.outer(volatility, volatility)
    return (
        pd.DataFrame(covariance, index=returns.columns, columns=returns.columns),
        float(fitted.shrinkage_),
    )

Fitted shrinkage intensity: 0.8556172839506178


,historical_return_vol,historical_return_variance
index,,
Index A,0.005774,0.000033
Index B,0.011547,0.000133
Index C,0.017321,0.000300


## 4. Replace historical volatilities with per-index forward estimates

Supply annualized decimal volatilities at a common forecast horizon:

$$D=\operatorname{diag}(\widehat\sigma_i),\qquad
\widehat\Sigma=D\widehat C D.$$

The synthetic values 8%, 14%, and 18% borrow numbers from our discussion; they
are not measurements of these indices. Here we intentionally treat them as
**raw IV**, so the calculations target *implied/model risk*. Later we discuss
calibrating them when the objective is realized volatility.

The covariance diagonal must equal the supplied IV squared. It is already
annualized: multiplying by trading days again would be an error. Percentage-point
quotes must first be divided by 100. Asset labels, rather than column position,
determine alignment. Local-currency IV needs FX adjustments if your actual holding
is unhedged in another base currency.

In [4]:
annualized_vol = pd.Series([0.08, 0.14, 0.18], index=assets, name='annual_forward_vol')
display(Code(inspect.getsource(forward_covariance), language='python'))
annual_covariance, fitted_shrinkage = forward_covariance(returns, annualized_vol)
assert np.allclose(np.diag(annual_covariance), annualized_vol ** 2)
reordered_covariance, _ = forward_covariance(returns, annualized_vol.iloc[::-1])
assert np.allclose(reordered_covariance, annual_covariance)
display(pd.DataFrame({
    'annual_forward_vol': annualized_vol,
    'annual_forward_variance': np.diag(annual_covariance),
}).style.format({'annual_forward_vol': '{:.1%}', 'annual_forward_variance': '{:.6f}'}))
display(annual_covariance)
print('Forward covariance eigenvalues:', np.linalg.eigvalsh(annual_covariance))

def forward_covariance(returns, annualized_vol):
    """Combine shrunk historical correlations with labeled forward vols."""
    historical, shrinkage = estimate_covariance(returns)
    if not isinstance(annualized_vol, pd.Series):
        raise TypeError("annualized_vol must be a pandas Series")
    if (not annualized_vol.index.is_unique
            or len(annualized_vol) != len(historical)
            or not historical.columns.isin(annualized_vol.index).all()):
        raise ValueError("Forward volatilities must match the return asset labels")
    volatility = annualized_vol.reindex(historical.columns).to_numpy(dtype=float)
    if not np.isfinite(volatility).all() or (volatility <= 0).any():
        raise ValueError("Forward volatilities must be positive and finite")
    values = historical.to_numpy()
    historical_vol = np.sqrt(np.diag(values))
    correlation = values / np.outer(historical_vol, historical_vol)
    covariance = correlation * np.outer(volatility, volatility)
    return pd.DataFrame(
        covariance, index=historical.index, columns=historical.columns
    ), shrinkage

,annual_forward_vol,annual_forward_variance
index,,
Index A,8.0%,0.006400
Index B,14.0%,0.019600
Index C,18.0%,0.032400


index,Index A,Index B,Index C
index,,,
Index A,0.006400,0.001455,0.0000
Index B,0.001455,0.019600,0.0000
Index C,0.000000,0.000000,0.0324


Forward covariance eigenvalues: [0.00624144 0.01975856 0.0324    ]


## 5. Calculate risk contributions, then compare three allocations

For weights (u), portfolio volatility is $\sigma_p=\sqrt{u^\top\Sigma u}$.
The marginal volatility contribution is $(\Sigma u)_i/\sigma_p$, and the total
contribution is

$$\mathrm{RC}_i=u_i\frac{(\Sigma u)_i}{\sigma_p},\qquad
\mathrm{share}_i=\frac{u_i(\Sigma u)_i}{u^\top\Sigma u}.$$

Shares sum to one. Full ERC makes them equal to $1/N$. Equal capital weights
and inverse-volatility weights do not generally do that when correlations differ.
All weights and volatilities below are conditional on this synthetic model.

In [5]:
def risk_report(weights, covariance):
    '''Show volatility contributions for a labeled allocation in a given model.'''
    aligned = weights.reindex(covariance.index).to_numpy()
    marginal_variance = covariance.to_numpy() @ aligned
    variance = aligned @ marginal_variance
    portfolio_vol = float(np.sqrt(variance))
    report = pd.DataFrame({
        'weight': aligned,
        'vol_contribution': aligned * marginal_variance / portfolio_vol,
        'risk_share': aligned * marginal_variance / variance,
    }, index=covariance.index)
    return report, portfolio_vol


erc_allocation = equal_risk_weights(annual_covariance)
inverse_vol = 1 / annualized_vol
inverse_vol /= inverse_vol.sum()
methods = {
    'Equal weight': pd.Series(1 / len(assets), index=assets),
    'Inverse volatility': inverse_vol,
    'Full ERC': erc_allocation['weight'],
}
reports, portfolio_vols = {}, {}
for name, weights in methods.items():
    reports[name], portfolio_vols[name] = risk_report(weights, annual_covariance)
    assert np.allclose(reports[name]['risk_share'].sum(), 1)
comparison = pd.concat(reports, names=['method', 'index'])
assert np.allclose(reports['Full ERC']['risk_share'], 1 / len(assets))
display(comparison.style.format('{:.4%}'))
display(pd.Series(portfolio_vols, name='annual_portfolio_forecast_vol').to_frame().style.format('{:.3%}'))

figure = make_subplots(rows=1, cols=2, subplot_titles=['Capital weights', 'Risk shares'])
for name, report in reports.items():
    for column, metric in [(1, 'weight'), (2, 'risk_share')]:
        figure.add_trace(go.Bar(
            x=assets.tolist(), y=report[metric], name=name,
            legendgroup=name, showlegend=(column == 1),
            hovertemplate='%{x}: %{y:.2%}<extra>' + name + '</extra>',
        ), row=1, col=column)
figure.update_layout(barmode='group', title='Equal capital, inverse volatility, and full ERC differ')
figure.update_yaxes(tickformat='.0%')
figure.show()

,annual_portfolio_forecast_vol
Equal weight,8.254%
Inverse volatility,7.165%
Full ERC,7.207%


## 6. What the ERC optimizer solves

In correlation units, solve for positive risk exposures $x$:

$$\min_{x>0}\left[\tfrac12x^\top\widehat Cx-\sum_i\log x_i\right].$$

The first-order conditions are $x_i(\widehat Cx)_i=1$ for every asset. Rescale
by volatility and normalize: $u_i\propto x_i/\widehat\sigma_i$. This produces
equal contribution shares. With positive definite correlation, the objective is
strictly convex in $x$. The implementation parameterizes $x=\exp(z)$ for
positivity; it does not claim convexity in $z$.

SciPy BFGS uses an analytical gradient. No covariance inverse, arbitrary
eigenvalue floor, or silent inverse-volatility fallback is used. Numerical
equalization is approximate, so inspect the achieved error.
[Risk-budgeting formulation](https://arxiv.org/pdf/1311.4057)

In [6]:
display(Code(inspect.getsource(equal_risk_weights), language='python'))
print('Largest absolute ERC risk-share error:',
      float((erc_allocation['risk_share'] - erc_allocation['target_risk_share']).abs().max()))

# In the diagonal-covariance special case, ERC is inverse-volatility sizing.
diagonal_covariance = pd.DataFrame(
    np.diag(annualized_vol ** 2), index=assets, columns=assets,
)
diagonal_erc = equal_risk_weights(diagonal_covariance)
assert np.allclose(diagonal_erc['weight'], inverse_vol)
display(pd.DataFrame({'inverse_vol': inverse_vol, 'diagonal_erc': diagonal_erc['weight']}).style.format('{:.3%}'))

def equal_risk_weights(covariance):
    """Return fully invested positive weights and variance-contribution shares."""
    if not isinstance(covariance, pd.DataFrame):
        raise TypeError("covariance must be a pandas DataFrame")
    if (covariance.empty or not covariance.index.is_unique
            or not covariance.index.equals(covariance.columns)):
        raise ValueError("Covariance must have matching, unique asset labels")
    values = covariance.to_numpy(dtype=float)
    if not np.isfinite(values).all() or (np.diag(values) <= 0).any():
        raise ValueError("Covariance must be finite with positive variances")
    volatility = np.sqrt(np.diag(values))
    correlation = values / np.outer(volatility, volatility)
    if not np.allclose(correlation, correlation.T):
        raise ValueError("Covariance must be symmetric")
    correlation = (correlation + correlation.T) / 2
    try:
        np.linalg.cholesky(correlation)
    except np.linalg.LinAlgError as error:
        raise ValueError("Covariance must be positive definite") from error

    def objective(log_x):
        x = np.exp(log_x)
        marginal = correlation @ x
        return 0.5 * x @ marginal - log_x.sum(), x * marginal - 1

    result = minimize(
        objective, np.zeros(len(volatility)), jac=True, method="BFGS"
    )
    if not result.success:
        raise RuntimeError(f"Risk-parity optimization failed: {result.message}")
    weights = np.exp(result.x) / volatility
    weights /= weights.sum()
    # Use the same symmetric matrix as the optimization, in original units.
    values = correlation * np.outer(volatility, volatility)
    contributions = weights * (values @ weights)
    variance = contributions.sum()
    if (not np.isfinite(weights).all() or (weights <= 0).any()
            or not np.isfinite(variance) or variance <= 0):
        raise RuntimeError("Optimization produced invalid weights or variance")
    return pd.DataFrame(
        {"weight": weights, "risk_share": contributions / variance,
         "target_risk_share": 1 / len(weights)},
        index=covariance.index,
    )

Largest absolute ERC risk-share error: 2.5654912261252605e-07


,inverse_vol,diagonal_erc
index,,
Index A,49.606%,49.606%
Index B,28.346%,28.346%
Index C,22.047%,22.047%


## 7. Apply one multiplier to all equity weights

The fully invested ERC mix $u$ sums to one. For an annual target $\sigma^*$,
use

$$a=\min\left(a_{\max},\frac{\sigma^*}{\widehat\sigma_p}\right),
\quad w=au,\quad c=1-a.$$

The cap is explicit: 1 means no leverage. If it binds, the target is unattainable
under that exposure policy. Negative cash means financed equity exposure, not an
extra asset to include in the equity ERC solve. Cash/financing is assumed to have
negligible price volatility; a risky reserve needs its own covariance terms.

Risk shares stay unchanged because both their numerator and denominator multiply
by $a^2$. Check this directly rather than relying only on the summary output.

In [7]:
TARGET_VOL = 0.08  # Illustration from the discussion, not a recommendation.
MAX_EXPOSURE = 1.0  # Illustrative no-leverage policy.
display(Code(inspect.getsource(volatility_target), language='python'))
allocation, summary = volatility_target(annual_covariance, TARGET_VOL, MAX_EXPOSURE)
scaled_risk, independently_measured_vol = risk_report(allocation['weight'], annual_covariance)
assert np.allclose(scaled_risk['risk_share'], allocation['risk_share'])
assert np.allclose(independently_measured_vol, summary['forecast_vol'])
assert np.allclose(allocation['weight'].sum() + summary['cash_weight'], 1)
display(allocation.style.format('{:.4%}'))
display(pd.Series(summary, name='value').to_frame().style.format('{:.4%}'))

def volatility_target(annualized_covariance, target_vol, max_exposure):
    """Return scaled ERC allocation and annual risk / financing summary."""
    target_vol, max_exposure = float(target_vol), float(max_exposure)
    if (not np.isfinite(target_vol) or target_vol <= 0
            or not np.isfinite(max_exposure) or max_exposure <= 0):
        raise ValueError("Target volatility and maximum exposure must be positive and finite")
    allocation = equal_risk_weights(annualized_covariance).rename(
        columns={"weight": "erc_weight"}
    )
    weights = allocation["erc_weight"].to_numpy()
    variance = weights @ annualized_covariance.to_numpy() @ weights
    if not np.isfinite(variance) or variance <= 0:
        raise ValueError("Forecast portfolio variance must be positive and finite")
    unscaled_vol = float(np.sqrt(variance))
    exposure = min(max_exposure, target_vol / unscaled_vol)
    allocation["weight"] = allocation["erc_weight"] * exposure
    return allocation, {
        "target_vol": target_vol,
        "unscaled_forecast_vol": unscaled_vol,
        "exposure": exposure,
        "cash_weight": 1 - exposure,
        "forecast_vol": exposure * unscaled_vol,
    }

,erc_weight,risk_share,target_risk_share,weight
index,,,,
Index A,48.9268%,33.3333%,33.3333%,48.9268%
Index B,27.9582%,33.3333%,33.3333%,27.9582%
Index C,23.1150%,33.3334%,33.3333%,23.1150%


,value
target_vol,8.0000%
unscaled_forecast_vol,7.2065%
exposure,100.0000%
cash_weight,0.0000%
forecast_vol,7.2065%


The fixture's fully invested forecast is about 7.21%, below the illustrative 8%
target. With the cap of 1, the portfolio stays fully invested and does **not**
reach 8%. The next cell demonstrates financing algebra by choosing a target
exactly twice this basket's forecast, and comparing caps of 1 and 2. The latter
allows exactly the exposure needed by that exercise; it is not a leverage
recommendation.

In [8]:
base_vol = summary['unscaled_forecast_vol']
financing_examples = []
for label, cap in [('No leverage', 1.0), ('Allow the required twofold exposure', 2.0)]:
    financed_allocation, financed_summary = volatility_target(annual_covariance, 2 * base_vol, cap)
    financing_examples.append({'policy': label, **financed_summary})
    assert np.allclose(financed_allocation['weight'].sum() + financed_summary['cash_weight'], 1)
financing_table = pd.DataFrame(financing_examples).set_index('policy')
assert np.allclose(financing_table.iloc[0]['forecast_vol'], base_vol)
assert np.allclose(financing_table.iloc[1]['cash_weight'], -1)
display(financing_table.style.format('{:.3%}'))

,target_vol,unscaled_forecast_vol,exposure,cash_weight,forecast_vol
policy,,,,,
No leverage,14.413%,7.207%,100.000%,0.000%,7.207%
Allow the required twofold exposure,14.413%,7.207%,200.000%,-100.000%,14.413%


## 8. Reproduce the 8%, 14%, and 18% portfolio scenarios

Hold correlation and relative IV ratios fixed. Multiply all IVs by a common
factor so that the fully invested basket's forecast becomes the specified
scenario value. This is a constructed sensitivity exercise, not a historical
simulation and not a way to infer real IV from a realized year.

For the 8% target, an unscaled forecast of 14% requires $8/14$ equity exposure;
18% requires $8/18$. The remainder is cash. These must be *contemporaneous
forecasts available before execution*, not eventual annual realized volatilities.

In [9]:
scenario_records = []
for scenario_vol in [0.08, 0.14, 0.18]:
    scenario_iv = annualized_vol * scenario_vol / base_vol
    scenario_covariance, _ = forward_covariance(returns, scenario_iv)
    scenario_allocation, scenario_summary = volatility_target(scenario_covariance, TARGET_VOL, MAX_EXPOSURE)
    assert np.allclose(scenario_summary['unscaled_forecast_vol'], scenario_vol)
    assert np.allclose(scenario_summary['forecast_vol'], TARGET_VOL)
    scenario_records.append(scenario_summary)
scenario_table = pd.DataFrame(scenario_records)
display(scenario_table.style.format('{:.3%}'))

figure = go.Figure()
for metric, label in [('exposure', 'Equities'), ('cash_weight', 'Cash')]:
    figure.add_bar(
        x=scenario_table['unscaled_forecast_vol'], y=scenario_table[metric], name=label,
        hovertemplate='Unscaled forecast %{x:.1%}; allocation %{y:.2%}<extra>' + label + '</extra>',
    )
figure.update_layout(barmode='stack', title='Higher forecast risk reduces total equity exposure')
figure.update_xaxes(title='Annual portfolio forecast before scaling', tickformat='.0%')
figure.update_yaxes(title='Allocation relative to NAV', tickformat='.0%')
figure.show()

,target_vol,unscaled_forecast_vol,exposure,cash_weight,forecast_vol
0,8.000%,8.000%,100.000%,0.000%,8.000%
1,8.000%,14.000%,57.143%,42.857%,8.000%
2,8.000%,18.000%,44.444%,55.556%,8.000%


## 9. Uniform IV shock: allocation versus total exposure

If every IV doubles while correlations stay fixed, covariance quadruples. ERC
weights do not change, portfolio forecast volatility doubles, and the **uncapped**
exposure multiplier halves. A binding cap can prevent the actual multiplier from
halving, because the original portfolio was already underexposed relative to its
target. Compare both cases below.

For the uncapped arithmetic check, the cap is derived as twice the exposure
required by this fixture, solely to keep it nonbinding.

In [10]:
doubled_covariance, _ = forward_covariance(returns, annualized_vol * 2)
nonbinding_cap = 2 * TARGET_VOL / base_vol
before, before_summary = volatility_target(annual_covariance, TARGET_VOL, nonbinding_cap)
after, after_summary = volatility_target(doubled_covariance, TARGET_VOL, nonbinding_cap)
assert np.allclose(doubled_covariance, annual_covariance * 4)
assert np.allclose(after['erc_weight'], before['erc_weight'])
assert np.allclose(after['weight'], before['weight'] / 2)
assert np.allclose(after_summary['forecast_vol'], TARGET_VOL)
display(pd.DataFrame({
    'ERC before': before['erc_weight'], 'ERC after': after['erc_weight'],
    'Exposure weight before': before['weight'], 'Exposure weight after': after['weight'],
}).style.format('{:.3%}'))
_, capped_after_summary = volatility_target(doubled_covariance, TARGET_VOL, MAX_EXPOSURE)
display(pd.DataFrame({
    'Before, nonbinding cap': before_summary,
    'After, nonbinding cap': after_summary,
    'Before, no leverage': summary,
    'After, no leverage': capped_after_summary,
}).T.style.format('{:.3%}'))

,ERC before,ERC after,Exposure weight before,Exposure weight after
index,,,,
Index A,48.927%,48.927%,54.314%,27.157%
Index B,27.958%,27.958%,31.036%,15.518%
Index C,23.115%,23.115%,25.660%,12.830%


,target_vol,unscaled_forecast_vol,exposure,cash_weight,forecast_vol
"Before, nonbinding cap",8.000%,7.207%,111.010%,-11.010%,8.000%
"After, nonbinding cap",8.000%,14.413%,55.505%,44.495%,8.000%
"Before, no leverage",8.000%,7.207%,100.000%,0.000%,7.207%
"After, no leverage",8.000%,14.413%,55.505%,44.495%,8.000%


## 10. A single index's IV changes the relative mix

Now double only Index A's IV. Its allocation within the ERC basket falls, while
the other indices' relative weights rise. The portfolio overlay separately
responds to the new basket's total forecast risk. Keep the two outputs distinct:
`erc_weight` describes the equity mix, while `weight` is exposure relative to NAV.

In [11]:
one_index_iv = annualized_vol.copy()
one_index_iv.loc['Index A'] *= 2
one_index_covariance, _ = forward_covariance(returns, one_index_iv)
one_index_allocation, one_index_summary = volatility_target(one_index_covariance, TARGET_VOL, MAX_EXPOSURE)
assert np.allclose(one_index_allocation['risk_share'], 1 / len(assets))
assert one_index_allocation.loc['Index A', 'erc_weight'] < allocation.loc['Index A', 'erc_weight']
display(pd.DataFrame({
    'IV before': annualized_vol, 'IV after': one_index_iv,
    'ERC before': allocation['erc_weight'], 'ERC after': one_index_allocation['erc_weight'],
    'NAV weight before': allocation['weight'], 'NAV weight after': one_index_allocation['weight'],
}).style.format('{:.3%}'))
display(pd.DataFrame({'Before': summary, 'After Index A IV shock': one_index_summary}).T.style.format('{:.3%}'))

,IV before,IV after,ERC before,ERC after,NAV weight before,NAV weight after
index,,,,,,
Index A,8.000%,16.000%,48.927%,32.386%,48.927%,27.157%
Index B,14.000%,14.000%,27.958%,37.013%,27.958%,31.036%
Index C,18.000%,18.000%,23.115%,30.601%,23.115%,25.660%


,target_vol,unscaled_forecast_vol,exposure,cash_weight,forecast_vol
Before,8.000%,7.207%,100.000%,0.000%,7.207%
After Index A IV shock,8.000%,9.540%,83.853%,16.147%,8.000%


## 11. Correlation can increase without IV changing

IV supplies marginal volatility, not cross-index correlation. For a synthetic
stress exercise, give **every pair** the same 0.9 correlation used earlier for
A/B. This is not a recommended stress threshold or a forecast. Keep IV unchanged.

First measure the existing ERC mix under this changed covariance. Then recompute
ERC and the overlay together. The frozen mix need not retain equal risk shares
under a new correlation matrix. This is why historical/shrunk correlation remains
an assumption, even when each index has a forward-looking IV quote.

In [12]:
stress_correlation = pd.DataFrame(0.9, index=assets, columns=assets)
np.fill_diagonal(stress_correlation.values, 1)
stress_covariance = stress_correlation * np.outer(annualized_vol, annualized_vol)
frozen_mix_report, frozen_mix_vol = risk_report(allocation['erc_weight'], stress_covariance)
stress_allocation, stress_summary = volatility_target(stress_covariance, TARGET_VOL, MAX_EXPOSURE)
assert np.allclose(np.diag(stress_covariance), np.diag(annual_covariance))
assert np.allclose(stress_allocation['risk_share'], 1 / len(assets))
assert stress_summary['unscaled_forecast_vol'] > summary['unscaled_forecast_vol']
display(pd.DataFrame({
    'Original model risk share': allocation['risk_share'],
    'Frozen mix under stress': frozen_mix_report['risk_share'],
    'Recomputed ERC under stress': stress_allocation['risk_share'],
}).style.format('{:.4%}'))
display(pd.DataFrame({'Original model': summary, 'Higher-correlation model': stress_summary}).T.style.format('{:.3%}'))
print('Frozen mix forecast under stress:', f'{frozen_mix_vol:.3%}')

,Original model risk share,Frozen mix under stress,Recomputed ERC under stress
index,,,
Index A,33.3333%,32.6229%,33.3333%
Index B,33.3333%,32.6229%,33.3333%
Index C,33.3334%,34.7542%,33.3333%


,target_vol,unscaled_forecast_vol,exposure,cash_weight,forecast_vol
Original model,8.000%,7.207%,100.000%,0.000%,7.207%
Higher-correlation model,8.000%,11.502%,69.554%,30.446%,8.000%


Frozen mix forecast under stress: 11.583%


## 12. Raw IV is not an unbiased realized-volatility forecast

Option prices contain risk premia. Cboe documents that SPX implied volatility
tends to exceed subsequent realized volatility. That does not authorize a fixed
discount for every index or every regime. [Cboe](https://www.cboe.com/tradable-products/vix)

For a realized-volatility objective, estimate a positive variance mapping using
your historical IV and forward realizations:

$$\widehat v_{i,t}=f_{i,t}(\mathrm{IV}_{i,t}^{2}),\qquad
\widehat\sigma_{i,t}=\sqrt{\widehat v_{i,t}}.$$

The model, calibration history, and IV horizon are caller choices. This notebook
does not invent a calibration function or fit one to synthetic outcomes. Supply
its out-of-sample forecasts to `forward_covariance` using the same interface.
Calibrating individual variances does not calibrate missing correlations.

The essential timing rule is that **a label's complete forward realization must
have finished and been available before fitting**. The next table contains
hand-written timestamp records only. Their common endpoint spacing is a teaching
fixture, not an IV-maturity recommendation. It also shows why an observation's
date alone does not establish whether its forward label is usable.

In [13]:
fit_at = pd.Timestamp('2026-09-09 16:00')
label_dates = pd.DataFrame({
    'observed_at': pd.to_datetime(['2026-09-01 16:00', '2026-09-02 16:00', '2026-09-03 16:00']),
    'label_end_at': pd.to_datetime(['2026-09-08 16:00', '2026-09-09 16:00', '2026-09-10 16:00']),
    'label_available_at': pd.to_datetime(['2026-09-08 18:00', '2026-09-09 18:00', '2026-09-10 18:00']),
})
label_dates['usable_at_fit'] = (
    (label_dates['label_end_at'] <= fit_at)
    & (label_dates['label_available_at'] <= fit_at)
)
eligible_label_records = label_dates.loc[label_dates['usable_at_fit']]
assert label_dates['usable_at_fit'].tolist() == [True, False, False]
print('Calibration timestamp:', fit_at)
display(label_dates)
display(eligible_label_records)

Calibration timestamp: 2026-09-09 16:00:00


,observed_at,label_end_at,label_available_at,usable_at_fit
0,2026-09-01 16:00:00,2026-09-08 16:00:00,2026-09-08 18:00:00,True
1,2026-09-02 16:00:00,2026-09-09 16:00:00,2026-09-09 18:00:00,False
2,2026-09-03 16:00:00,2026-09-10 16:00:00,2026-09-10 18:00:00,False


,observed_at,label_end_at,label_available_at,usable_at_fit
0,2026-09-01 16:00:00,2026-09-08 16:00:00,2026-09-08 18:00:00,True


## 13. Execute the guide's caller-data procedure on the fixture

This is the guide's price-window workflow, using the data already constructed.
The window includes one extra price observation to produce the required number
of returns. IV is selected at the signal timestamp. An IV provider's release time
must be checked separately; date-label alignment alone cannot prove availability.

Keep the same target and exposure policy when evaluating both allocation and
risk. The forward covariance is already annualized.

In [14]:
forward_vols = annualized_vol.to_frame().T
forward_vols.index = pd.DatetimeIndex([signal_date])
history = total_return_prices.loc[:signal_date].tail(correlation_lookback + 1)
if len(history) != correlation_lookback + 1:
    raise ValueError('Insufficient history for the chosen correlation lookback')
window_returns = history.pct_change(fill_method=None).iloc[1:]
snapshot_vol = forward_vols.loc[signal_date]
window_covariance, window_shrinkage = forward_covariance(window_returns, snapshot_vol)
window_allocation, window_summary = volatility_target(window_covariance, TARGET_VOL, MAX_EXPOSURE)
assert np.allclose(window_covariance, annual_covariance)
assert np.allclose(window_allocation['weight'], allocation['weight'])
display(window_allocation.style.format('{:.4%}'))

,erc_weight,risk_share,target_risk_share,weight
index,,,,
Index A,48.9268%,33.3333%,33.3333%,48.9268%
Index B,27.9582%,33.3333%,33.3333%,27.9582%
Index C,23.1150%,33.3334%,33.3333%,23.1150%


## 14. Substitute your own data without changing the solver

Set the variables below to run on actual data. Both CSVs must have a date column
first and one column per index. Prices must be adjusted total-return prices;
forward volatilities must already be annualized decimals, with matching asset
labels. The volatility panel may contain raw IV for an implied-risk objective or
calibrated forecasts for a realized-risk objective.

All owner choices start unset: this cell does not silently reuse synthetic data,
select a lookback, choose an IV maturity, or assign a target/leverage limit. Use
paths relative to the repository root or absolute paths. Check currencies,
economic exposures, source timestamps, and the absence of stale quotes before
calling the functions.

In [15]:
PRICE_CSV = None
FORWARD_VOL_CSV = None
USER_SIGNAL_DATE = None
USER_CORRELATION_LOOKBACK = None
USER_TARGET_VOL = None
USER_MAX_EXPOSURE = None

inputs = [PRICE_CSV, FORWARD_VOL_CSV, USER_SIGNAL_DATE, USER_CORRELATION_LOOKBACK,
          USER_TARGET_VOL, USER_MAX_EXPOSURE]
if all(value is None for value in inputs):
    print('Own-data example is unset. Fill all six inputs to run it.')
elif any(value is None for value in inputs):
    raise ValueError('Supply both data files, signal date, lookback, target, and exposure cap')
else:
    if not isinstance(USER_CORRELATION_LOOKBACK, int) or USER_CORRELATION_LOOKBACK < 2:
        raise ValueError('Lookback must be an integer with at least two returns')
    user_prices = pd.read_csv(ROOT / PRICE_CSV, index_col=0, parse_dates=True)
    user_vols = pd.read_csv(ROOT / FORWARD_VOL_CSV, index_col=0, parse_dates=True)
    if not user_prices.index.is_unique or not user_prices.index.is_monotonic_increasing:
        raise ValueError('Prices must have unique, sorted dates')
    if not user_vols.index.is_unique or not user_vols.index.is_monotonic_increasing:
        raise ValueError('Forward-volatility dates must be unique and sorted')
    selected_date = pd.Timestamp(USER_SIGNAL_DATE)
    user_history = user_prices.loc[:selected_date].tail(USER_CORRELATION_LOOKBACK + 1)
    if len(user_history) != USER_CORRELATION_LOOKBACK + 1:
        raise ValueError('Insufficient history for the selected lookback')
    if not np.isfinite(user_history.to_numpy(dtype=float)).all() or (user_history <= 0).any().any():
        raise ValueError('Selected prices must be positive and finite')
    user_returns = user_history.pct_change(fill_method=None).iloc[1:]
    user_covariance, user_shrinkage = forward_covariance(user_returns, user_vols.loc[selected_date])
    user_allocation, user_summary = volatility_target(user_covariance, USER_TARGET_VOL, USER_MAX_EXPOSURE)
    print('Correlation shrinkage:', user_shrinkage)
    display(user_allocation.style.format('{:.4%}'))
    display(pd.Series(user_summary, name='value').to_frame().style.format('{:.4%}'))

Own-data example is unset. Fill all six inputs to run it.


## 15. Input errors should fail visibly

Missing returns, missing asset forecasts, and nonpositive IV are not silently
filled. Estimate correlation from one common observation panel rather than using
pairwise covariance with different dates for every pair. Inspect corporate
actions and quote errors instead of treating them as market risk.

The code validates labels and numbers. It cannot establish that an apparently
finite quote is current, or that currencies and maturities match. Those are
data-provenance checks for your real input pipeline.

In [16]:
bad_returns = returns.copy()
bad_returns.iloc[0, 0] = np.nan
bad_labels = annualized_vol.rename(index={'Index A': 'Unknown index'})
for label, args in [
    ('Missing return', (bad_returns, annualized_vol)),
    ('Wrong index label', (returns, bad_labels)),
    ('Zero forward volatility', (returns, annualized_vol * 0)),
]:
    try:
        forward_covariance(*args)
    except ValueError as error:
        print(f'{label}: {error}')
    else:
        raise AssertionError(f'{label} was not rejected')

Missing return: Use a complete, finite, aligned return panel
Wrong index label: Forward volatilities must match the return asset labels
Zero forward volatility: Forward volatilities must be positive and finite


## 16. Execution, accounting, and walk-forward evaluation

Compute weights from information available at the signal time, then execute
with the configured positive business-day lag. Apollo3 signals never execute on
the close that produced them; the actual fill also requires valid exact-date
prices for every held/targeted name. A forecast at a foreign index's close may
not be available at another exchange's earlier close on the same calendar date.

Always forecast the **unscaled current equity mix**. A volatility estimate from
past *targeted* portfolio returns confounds changes in allocation with prior
exposure multipliers. If the ERC mix is frozen while covariance changes, recompute
its forecast risk but do not assume its risk shares remain equal. Section 11
demonstrated that distinction.

For weights actually in force over a return interval, cash (c=1-\sum_i w_i$,
and costs measured relative to NAV, an accounting sketch is

$$r_p=\sum_iw_ir_i+\max(c,0)r_{\mathrm{cash}}
-\max(-c,0)r_{\mathrm{borrow}}-\mathrm{costs}.$$

A real backtest must also respect weight drift between fills, trade availability,
instrument mechanics, and the chosen rebalancing schedule. This walkthrough is
not a historical performance backtest.

Choose correlation lookback, calibration history, and rebalance cadence using
walk-forward evidence. Evaluate forward variance forecasts, achieved risk shares,
turnover, realized volatility, drawdowns, and net returns including cash interest,
financing, and transaction costs. Do not infer the optimal choices from this toy
panel or tune against future crisis results.

The target is a **forecast**, not a guarantee of realized volatility or maximum
loss. A jump can happen before the overlay reacts; reducing exposure after a
selloff can miss part of a fast rebound. Correlation misspecification and execution
lag remain relevant even with per-index IV.
[Volatility-targeting timing tradeoffs](https://www.man.com/insights/volatility-is-back-better-to-target-returns-or-target-risk)

The runnable cells have checked covariance construction, label alignment, ERC
shares, cap/cash accounting, crisis scenarios, shock scaling, and mature-label
filtering. The source recipe's independent analytical self-check remains available:

```bash
./venv/bin/python docs/research/risk_parity_recipe.py
```

Return to [the guide](../../docs/research/erc_vol_targetting.md) or
[the runnable recipe](../../docs/research/risk_parity_recipe.py) when wiring this
into your own research. A realized-risk target needs your calibrated forecasts;
a raw-IV target needs no such reinterpretation, but means something different.